O código disponibilizado aqui foi produzido principalmente com base nos seguintes tutoriais:
1. TensorFlow - [Load Video Data](https://www.tensorflow.org/tutorials/load_data/video) (Extração e processamento dos frames dos vídeos)
2. TensorFlow - [Video classification with a 3D convolutional neural network](https://www.tensorflow.org/tutorials/video/video_classification) (Estrutura geral de uma CNN 2D+1)
3. GeeksForGeeks - [Video Classification with a 3D Convolutional Neural Network](https://www.geeksforgeeks.org/) (Adaptação do código anterior para um modelo de CNN 3D, como visto na disciplina)

Adaptações nos códigos foram realizadas com objetivo de adaptá-los à estrutura do dataset trabalhado. Os comentários em inglês são mantidos conforme presente nos códigos originais. Já os comentários em português foram inseridos para facilitar a compreensão de algumas etapas.


In [ ]:
# Bibliotecas voltadas para o processamento de dados
import numpy as np
import random
import itertools
import kagglehub
import os
import cv2

# Bibliotecas voltadas para execução da rede
import tensorflow as tf
import keras
from keras.models import Sequential
from keras.layers import Conv3D, MaxPooling3D, Flatten, Dense, Dropout, BatchNormalization
from collections import Counter

# Bibliotecas voltadas para análise de dados
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

## Procesamento de dados

In [ ]:
# Download latest version
path = kagglehub.dataset_download("twonzii/surfing-maneuver-classification")
path = os.path.join(path, "data-split")

print("Path to dataset files:", path)

In [ ]:
def get_subset_files(source_dir, percentage, seed=42):
    """
    Picks a percentage of the videos and returns their paths.

    Args:
      source_dir: Folder with the full dataset
      percentage: How much of the data to use, from 1 to 100
      seed: Any number. Same seed = same videos picked every time.

    Return:
      A dictionary where each item is (path to the video, class name)
      {
         "train": [("data-split/train/roller/video1.mp4", "roller"), ...],
          "val":   [...],
          "test":  [...],
      }
    """

    if percentage <= 0 or percentage > 100:
        raise ValueError("Percentage must be between 1 and 100")

    random.seed(seed)

    subset = {}

    for split in ["train", "val", "test"]:
        split_path = os.path.join(source_dir, split)
        subset[split] = []

        if not os.path.isdir(split_path):
            print(f"Folder not found, skipping: {split_path}")
            continue

        for class_name in sorted(os.listdir(split_path)):
            class_path = os.path.join(split_path, class_name)

            if not os.path.isdir(class_path):
                continue

            videos = sorted(f for f in os.listdir(class_path)
                            if f.lower().endswith(".mp4"))

            if len(videos) == 0:
                continue

            # Mantém, no mínimo, um vídeo por classe
            how_many = max(1, round(len(videos) * percentage / 100))

            chosen = random.sample(videos, how_many)

            for video in chosen:
                subset[split].append((os.path.join(class_path, video), class_name))

            print(f"{split}/{class_name}: kept {how_many} of {len(videos)} videos")

    return subset

In [ ]:
def format_frames(frame, output_size):
  """
    Pad and resize an image from a video.

    Args:
      frame: Image that needs to resized and padded.
      output_size: Pixel size of the output frame image.

    Return:
      Formatted frame with padding of specified output size.
  """
  frame = tf.image.convert_image_dtype(frame, tf.float32)
  frame = tf.image.resize_with_pad(frame, *output_size)
  return frame

In [ ]:
def frames_from_video_file(video_path, n_frames, output_size = (128,128), frame_step = 2):
  """
    Creates frames from each video file present for each category.

    Args:
      video_path: File path to the video.
      n_frames: Number of frames to be created per video file.
      output_size: Pixel size of the output frame image.

    Return:
      An NumPy array of frames in the shape of (n_frames, height, width, channels).
  """

  # Read each video frame by frame
  result = []
  src = cv2.VideoCapture(str(video_path))

  if not src.isOpened():
    raise ValueError(f"Could not open video: {video_path}")

  video_length = int(src.get(cv2.CAP_PROP_FRAME_COUNT))
  need_length = 1 + (n_frames - 1) * frame_step

  if need_length > video_length:
    start = 0
  else:
    max_start = video_length - need_length
    start = random.randint(0, max_start)

  src.set(cv2.CAP_PROP_POS_FRAMES, start)

  # ret is a boolean indicating whether read was successful, frame is the image itself
  ret, frame = src.read()
  if not ret:
    raise ValueError(f"Could not read the first frame of: {video_path}")
  result.append(format_frames(frame, output_size))

  for _ in range(n_frames - 1):
    for _ in range(frame_step):
      ret, frame = src.read()
    if ret:
      frame = format_frames(frame, output_size)
      result.append(frame)
    else:
      result.append(np.zeros_like(result[0]))
  src.release()
  result = np.array(result)[..., [2, 1, 0]]

  return result

In [ ]:
class FrameGenerator:
  def __init__(self, pairs, n_frames, training=False):
    """ Returns a set of frames with their associated label.

      Args:
        pairs: List of (video_path, class_name), like files["train"].
        n_frames: Number of frames.
        training: Boolean to determine if training dataset is being created.
    """
    self.pairs = list(pairs)
    self.n_frames = n_frames
    self.training = training
    self.class_names = sorted(set(name for _, name in self.pairs))
    self.class_ids_for_name = dict((name, idx) for idx, name in enumerate(self.class_names))

  def __call__(self):
    pairs = list(self.pairs)

    if self.training:
      random.shuffle(pairs)

    for path, name in pairs:
      video_frames = frames_from_video_file(path, self.n_frames)
      label = self.class_ids_for_name[name]   # Encode labels
      yield video_frames, label

In [ ]:
# Durante testes iniciais, foi utilizado um subset com 70% do dataset.
# Todos os dados finais foram obtidos com 100% do dataset.

files = get_subset_files(path, percentage=100)

In [ ]:
# Create the training set
output_signature = (tf.TensorSpec(shape = (None, None, None, 3), dtype = tf.float32),
                    tf.TensorSpec(shape = (), dtype = tf.int16))
train_ds = tf.data.Dataset.from_generator(FrameGenerator(files['train'], 16, training=True),
                                          output_signature = output_signature)

# Create the validation set
val_ds = tf.data.Dataset.from_generator(FrameGenerator(files['val'], 16),
                                        output_signature = output_signature)

# Create the test set
test_ds = tf.data.Dataset.from_generator(FrameGenerator(files['test'], 16),
                                        output_signature = output_signature)

In [ ]:
# Print the shapes of the data
train_frames, train_labels = next(iter(train_ds))
print(f'Shape of training set of frames: {train_frames.shape}')
print(f'Shape of training labels: {train_labels.shape}')

val_frames, val_labels = next(iter(val_ds))
print(f'Shape of validation set of frames: {val_frames.shape}')
print(f'Shape of validation labels: {val_labels.shape}')

test_frames, test_labels = next(iter(test_ds))
print(f'Shape of test set of frames: {test_frames.shape}')
print(f'Shape of test labels: {test_labels.shape}')

In [ ]:
# Data augmentation
# Código produzido com base na metodologia do artigo (NOME, ANO)

mean = float(np.mean([tf.reduce_mean(frames) for frames, _ in train_ds.take(100)]))
size = 128
crop = 112

def preprocess_train(frames, label):
    # Janela de recorte aleatória
    frames = tf.image.random_crop(frames, (16, crop, crop, 3))
    # Flip horizontal aleatório
    frames = tf.cond(tf.random.uniform(()) < 0.5,
                     lambda: tf.reverse(frames, axis=[2]),
                     lambda: frames)
    return frames - mean, label

def preprocess_eval(frames, label):
    frames = tf.image.central_crop(frames, crop / size)   # Recorte central (determinístico)
    return frames - mean, label

In [ ]:
# Criação dos datasets que serão fornecidos para rede
# Embaralha dataset de treino, aplica augmentação e outras funções
# para melhorar a perfomance, como cache() e prefetch()

AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).map(preprocess_train, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
val_ds   = val_ds.cache().map(preprocess_eval, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
test_ds  = test_ds.cache().map(preprocess_eval, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

In [ ]:
# Separata cada set em batches

train_ds = train_ds.batch(8)
val_ds = val_ds.batch(8)
test_ds = test_ds.batch(8)

# A entrada da CNN 3D possui cinco dimensões:
# batch_size, number_of_frames, height, width, channels

train_frames, train_labels = next(iter(train_ds))
print(f'Shape of training set of frames: {train_frames.shape}')
print(f'Shape of training labels: {train_labels.shape}')

val_frames, val_labels = next(iter(val_ds))
print(f'Shape of validation set of frames: {val_frames.shape}')
print(f'Shape of validation labels: {val_labels.shape}')

test_frames, test_labels = next(iter(test_ds))
print(f'Shape of test set of frames: {test_frames.shape}')
print(f'Shape of test labels: {test_labels.shape}')

## Construção da rede

In [ ]:
# Cria uma função personalizada para calcular métrica de F1-score
# com labels na forma "Sparse"

class SparseCategoricalF1Score(keras.metrics.F1Score):
    def update_state(self, y_true, y_pred, sample_weight=None):
        y_true = tf.one_hot(
            tf.cast(tf.reshape(y_true, [-1]), tf.int32),
            depth=tf.shape(y_pred)[-1]
        )

        return super().update_state(y_true, y_pred, sample_weight)

In [ ]:
def create_advanced_3dcnn_model(input_shape, num_classes):
    """
    Create an advanced 3D CNN model for video classification.

    Args:
    - input_shape (tuple): Shape of input frames (e.g., (16, 112, 112, 3) for 16 frames of size 112x112 pixels and 3 channels).
    - num_classes (int): Number of classes in the classification task.

    Returns:
    - model (Sequential): Compiled 3D CNN model.
    """
    model = Sequential()

    # 3D convolutional layer with 64 filters, kernel size of (3, 3, 3), and ReLU activation
    model.add(Conv3D(64, (3, 3, 3), activation='relu', padding='same', input_shape=input_shape))
    # 3D max pooling layer with pool size of (2, 2, 2)
    model.add(MaxPooling3D((2, 2, 2)))
    # Batch normalization layer
    model.add(BatchNormalization())

    # 3D convolutional layer with 128 filters
    model.add(Conv3D(128, (3, 3, 3), activation='relu', padding='same'))
    model.add(MaxPooling3D((2, 2, 2)))
    model.add(BatchNormalization())

    # 3D convolutional layer with 256 filters
    model.add(Conv3D(256, (3, 3, 3), activation='relu', padding='same'))
    model.add(MaxPooling3D((2, 2, 2)))
    model.add(BatchNormalization())

    # Flatten layer to flatten the output of the convolutional layers
    model.add(Flatten())
    # Fully connected (dense) layer with 512 units and ReLU activation
    model.add(Dense(512, activation='relu'))
    # Dropout layer
    model.add(Dropout(0.5))
    # Output layer with softmax activation for multi-class classification
    model.add(Dense(num_classes, activation='softmax'))

    # Compile the model with Adam optimizer, categorical crossentropy loss, and accuracy/F1-score metric
    model.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    optimizer=keras.optimizers.Adam(learning_rate=0.0001),
    metrics=[
        'accuracy',
        SparseCategoricalF1Score(
            average='macro',
            name='f1_score'
        )
    ])

    return model

# Define the shape of input frames and create the advanced 3D CNN model
class_names = sorted(set(name for _, name in files['train']))
num_classes = len(class_names)

input_shape = (None, 16, train_frames.shape[2], train_frames.shape[3], 3)
model = create_advanced_3dcnn_model(input_shape[1:], num_classes)

In [ ]:
# Código para salvar o melhor modelo
# Monitora F1-score, ideal para classes desbalanceadas

callbacks_list = [
    keras.callbacks.ModelCheckpoint(
        filepath='model.keras',
        monitor="val_f1_score",
        mode="max",
        save_best_only=True,
        verbose=1
    )
]

In [ ]:
# Calcula pesos com base no número de observações por classe
# Alternativa para combater desbalanceamento

counts = Counter(name for _, name in files["train"])

# O peso é o inverso da raiz quadrada do número de observações
# Quanto mais observações, menor peso

weights = np.array([
    1 / np.sqrt(counts[name])
    for name in class_names
])

# Normalização dos pesos
weights /= weights.mean()

class_weight = {
    i: float(weight)
    for i, weight in enumerate(weights)
}

In [ ]:
# Treinamento do modelo  com base nas variáveis definidas anteriormente
history = model.fit(x = train_ds,
                    epochs = 100,
                    validation_data = val_ds,
                    callbacks = callbacks_list,
                    class_weight=class_weight)

## Análise e validação do modelo

In [ ]:
def plot_history(history, figsize=(10, 4.5)):
    """
    Plota as curvas de aprendizado (loss e F1-score) lado a lado.

    Args:
      history: histórico do modelo com todas as métricas
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize)

    # Loss
    ax1.plot(history.history['loss'], label='Treino')
    ax1.plot(history.history['val_loss'], label='Validação')
    max_loss = max(history.history['loss'] + history.history['val_loss'])
    ax1.set_ylim([0, np.ceil(max_loss)])
    ax1.set_xlabel('Época')
    ax1.set_ylabel('Loss')
    ax1.legend()

    # F1-score
    ax2.plot(history.history['f1_score'], label='Treino')
    ax2.plot(history.history['val_f1_score'], label='Validação')
    ax2.set_ylim([0, 1])
    ax2.set_xlabel('Época')
    ax2.set_ylabel('F1-score')
    ax2.legend()

    fig.tight_layout()
    plt.show()

plot_history(history)

In [ ]:
model.load_weights('model.keras') # Carrega os pesos do melhor modelo

test_results = model.evaluate(test_ds, return_dict=True) # Avalia no conjunto de treino

print(f"Test loss: {test_results['loss']:.4f}")
print(f"Test accuracy: {test_results['accuracy']:.4f}")
print(f"Test loss: {test_results['f1_score']:.4f}")

y_true = np.concatenate([labels.numpy() for _, labels in test_ds])
y_pred = np.argmax(model.predict(test_ds, verbose=0), axis=1)

In [ ]:
# Plot da matriz de confusão nos dados de teste

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=np.arange(len(class_names))
)

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names
)

plt.xlabel('Classe predita')
plt.ylabel('Classe verdadeira')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Relatório de classificação com métricas por classe

report = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)
accuracy = report.pop('accuracy')

df = pd.DataFrame(report).T
display(
    df.style
    .format({'precision': '{:.2f}', 'recall': '{:.2f}', 'f1-score': '{:.2f}', 'support': '{:.0f}'})
)
print(f'Acurácia: {accuracy:.2%}')

In [ ]:
# Gráfico de barras com distribuição dos vídeos por classe

rows = []
nomes_conjuntos = {'train': 'Treino', 'val': 'Validação', 'test': 'Teste'}

for split in ['train', 'val', 'test']:
    counts = Counter(name for _, name in files[split])
    for name in class_names:
        rows.append({'Conjunto': nomes_conjuntos[split], 'Classe': name, 'Vídeos': counts[name]})

df_counts = pd.DataFrame(rows)

plt.figure(figsize=(10, 6))
ax = sns.barplot(data=df_counts, x='Classe', y='Vídeos', hue='Conjunto')
for container in ax.containers:
    ax.bar_label(container)

ax.set_xlabel('Classe')
ax.set_ylabel('Número de vídeos')
ax.legend(title='Conjunto')

plt.show()

In [ ]:
# Mapeamento de vídeos incorretamente classificados para verificação manual

errors = [(path, class_names[t], class_names[p])
          for (path, _), t, p in zip(files['test'], y_true, y_pred) if t != p]

print(f'{len(errors)} erros de {len(y_true)} vídeos')
for path, real, pred in errors:
    print(f'{os.path.basename(path)}: real = {real} | previsto = {pred}')